In [1]:

import sqlite3

import csv

import urllib.request

import os





BASE_URL = "https://raw.githubusercontent.com/Lioneln5/CIS3120/main/"

BOOK_URL   = BASE_URL + "Book.csv"

MEMBER_URL = BASE_URL + "Member.csv"

LOAN_URL   = BASE_URL + "Loan.csv"




BOOK_PATH   = "/content/Book.csv"

MEMBER_PATH = "/content/Member.csv"

LOAN_PATH   = "/content/Loan.csv"



DB_PATH = "/content/library.db"

In [2]:
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

In [3]:
for url, path in [(BOOK_URL, BOOK_PATH), (MEMBER_URL, MEMBER_PATH), (LOAN_URL, LOAN_PATH)]:

    urllib.request.urlretrieve(url, path)

    print(f"Downloaded: {path}")

Downloaded: /content/Book.csv
Downloaded: /content/Member.csv
Downloaded: /content/Loan.csv


In [4]:
conn = sqlite3.connect(DB_PATH)
conn.execute('PRAGMA foreign_keys = ON;')

conn.execute('''CREATE TABLE IF NOT EXISTS Book (
    callNo  TEXT    NOT NULL,
    title   TEXT    NOT NULL,
    author  TEXT    NOT NULL,
    PRIMARY KEY (callNo)
);''')

conn.execute('''CREATE TABLE IF NOT EXISTS Member (
    id          INTEGER NOT NULL,
    firstname   TEXT    NOT NULL,
    lastName    TEXT    NOT NULL,
    PRIMARY KEY (id)
);''')

conn.execute('''CREATE TABLE IF NOT EXISTS Loan (
    callNo        TEXT    NOT NULL,
    id            INTEGER NOT NULL,
    dateBorrowed  TEXT    NOT NULL,
    dateReturned  TEXT,
    dateDue       TEXT    NOT NULL,
    PRIMARY KEY (callNo, id, dateBorrowed),
    FOREIGN KEY (callNo) REFERENCES Book(callNo),
    FOREIGN KEY (id)     REFERENCES Member(id)
);''')

conn.commit()
print("Check")

Check


In [5]:
with open(BOOK_PATH, newline='', encoding='utf-8') as f:

    reader = csv.DictReader(f)

    for row in reader:

        conn.execute(

            'INSERT INTO Book (callNo, title, author) VALUES (?, ?, ?);',

            (row['callNo'], row['title'], row['author'])

        )



conn.commit()

print('Book rows loaded:', conn.execute('SELECT COUNT(*) FROM Book;').fetchone()[0])

Book rows loaded: 11


In [6]:
with open(MEMBER_PATH, newline='', encoding='utf-8') as f:

    reader = csv.DictReader(f)

    for row in reader:

        conn.execute(

            'INSERT INTO Member (id, firstname, lastName) VALUES (?, ?, ?);',

            (int(row['id']), row['firstname'], row['lastName'])

        )



conn.commit()

print('Member rows loaded:', conn.execute('SELECT COUNT(*) FROM Member;').fetchone()[0])

Member rows loaded: 4


In [7]:
with open(LOAN_PATH, newline='', encoding='utf-8') as f:

    reader = csv.DictReader(f)

    for row in reader:


        date_returned = row['dateReturned'] if row['dateReturned'].strip() else None



        conn.execute(

            '''INSERT INTO Loan (callNo, id, dateBorrowed, dateReturned, dateDue)

               VALUES (?, ?, ?, ?, ?);''',

            (row['callNo'], int(row['id']),

             row['dateBorrowed'], date_returned, row['dateDue'])

        )



conn.commit()

print('Loan rows loaded:', conn.execute('SELECT COUNT(*) FROM Loan;').fetchone()[0])

Loan rows loaded: 4


In [8]:
query1 = "SELECT * FROM Book ORDER BY author;"
for row in conn.execute(query1):
    print(row)

('R 487 T35 1967', 'Medicine in medieval England.', 'Charles H Talbot')
('QA 76.9 D26H39 1996', 'Data model patterns : conventions of thought', 'David Hay')
('CB 351 M293 1983', 'Atlas of medieval Europe', 'Donald Matthew')
('HQ 1143 P68 1975', 'Medieval women', 'Eileen Power')
('PC 14 V48 1965', 'Medieval miscellany', 'Frederick Whitehead')
('QA 76.73 S67C435 2004', "Joe Celko's Trees and hierarchies in SQL for smarties", 'Joe Celko')
('QA 76.73 S67C46 1997', "Joe Celko's SQL puzzles & answers", 'Joe Celko')
('QA 76.9 D35C45 1999', "Joe Celko's data & databases : concepts in practice", 'Joe Celko')
('R 141 E45 2006', 'Medieval medicine and the plague', 'Lynne Elliott')
('QA 76.9 D26H355 2008', 'Information modeling and relational databases', 'T A Halpin')
('QA 76.76 A65P76 2011', 'Programming Android', 'Zigurd R Mednieks')


Retrieve all columns from the Book table, ordered alphabetically by author last name



In [9]:
query2 = """
SELECT Book.title, Member.firstname, Member.lastName
FROM Loan
JOIN Book ON Loan.callNo = Book.callNo
JOIN Member ON Loan.id = Member.id
WHERE Loan.dateReturned IS NULL;
"""
for row in conn.execute(query2):
    print(row)

("Joe Celko's SQL puzzles & answers", 'David', 'Martin')
('Medieval medicine and the plague', 'David', 'Martin')


Retrieve the title of each book, and the first and last name of the member who borrowed it, for all loans where dateReturned is NULL.

In [10]:
query3 = """
SELECT Member.firstname || ' ' || Member.lastName AS fullName, dateBorrowed, dateDue, dateReturned
FROM Loan
JOIN Member ON Loan.id = Member.id
WHERE callNo = 'R 141 E45 2006'
ORDER BY dateBorrowed ASC;
"""
for row in conn.execute(query3):
    print(row)

('Betty Freeman', '4/1/2014 0:00', '4/15/2014 0:00', '4/15/2014 0:00')
('David Martin', '4/30/2014 0:00', '5/14/2014 0:00', None)


Retrieve the full loan history for the book with call No R 141 E45 2006 — showing the member's full name, dateBorrowed, dateDue, and dateReturned. Order by dateBorrowed ascending.

In [11]:
query4 = """
SELECT id, firstname, lastName
FROM Member
WHERE id NOT IN (SELECT id FROM Loan);
"""
for row in conn.execute(query4):
    print(row)

(4, 'John', 'Martin')


Retrieve the id, firstname, and lastName of every member who does not appear in the Loan table. Use a LEFT JOIN or a sub-query with NOT IN.

In [12]:
query5 = """
SELECT Member.firstname || ' ' || Member.lastName AS fullName, COUNT(Loan.id) as loanCount
FROM Member
LEFT JOIN Loan ON Member.id = Loan.id
GROUP BY Member.id
ORDER BY loanCount DESC;
"""
for row in conn.execute(query5):
    print(row)

('David Martin', 2)
('John Smith', 1)
('Betty Freeman', 1)
('John Martin', 0)


Retrieve each member's full name and the total number of loans they have made (including completed ones). Include members with zero loans. Order by number of loans descending.

Query 6

This query identifies the book that has been borrowed the most based on the number of loan records.

The business question being answered is "which book is the most popular among library members?"

This is useful because it helps the library understand which books are in the highest demand. With this information, the library can decide to purchase additional copies of popular titles, adjust inventory size, etc...

In [13]:
query6 = """
SELECT Book.title, COUNT(Loan.callNo) as borrowCount
FROM Book
JOIN Loan ON Book.callNo = Loan.callNo
GROUP BY Book.callNo
ORDER BY borrowCount DESC
LIMIT 1;
"""
for row in conn.execute(query6):
    print(row)

('Medieval medicine and the plague', 2)


Write one original query of your own design that reveals something interesting or useful about this library dataset. It must use at least one JOIN and at least one aggregate function or WHERE condition not used in Queries 1–5. Precede the query with a Markdown cell that states the business question you are answering and why it is useful.

In [14]:
conn.close()

A data quality issue in this dataset is that datereturned is blank for books that have not been returned, so it must be converted to null when loading into SQLite. Another issue is that the assignment’s expected result for Query 4 appears inconsistent with the actual CSV data, since only one member has no loan record. A limitation of this dataset is that it is very small and does not include additional library details such as book categories, fines, or multiple copies of the same title. Because of that, it does not fully represent the realistic complexity of a real library db.
